# 91 Charge / discharge LODES

8 h / 100 h / 300 h thickness; asymmetric Lc/Ld.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path
setup_path()
from plant_sim.params import default_params, CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G
from plant_sim.plotting import apply_style, timeseries, xy_plot, FIGURES
apply_style()
P = default_params()
print("6 M KOH default", P.c_KOH_mol_m3/1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G,2), round(CAPACITY_MAGNETITE_MAH_G,2))
print("figures ->", FIGURES)


In [ ]:
from plant_sim.plant import PlantModel
from plant_sim.scenarios.mission import charge_discharge_inputs
for hrs, span in [(8, 1200), (100, 1200)]:
    pp = P.with_duration(hrs, hrs)
    plant = PlantModel(pp)
    r = plant.simulate((0, span), inputs=charge_discharge_inputs(span/3, span/3, pp), n_eval=100, rtol=1e-4)
    timeseries(r["t"]/60, {"I": r["alg"]["I_cell_A"], "V": r["alg"]["V_cell_V"], "SOC": r["alg"]["SOC"]}, xlabel="t (min)", ylabel="I/V/SOC", title=f"{hrs} h thickness case", callout="US LODES duration scaling", stem=f"nb91_{hrs}h")
    print(hrs, "h L_anode_cm", round(pp.L_anode_m*100, 2), "finite", np.all(np.isfinite(r["y"])))
pp = P.with_duration(100, 300)
print("asymmetric 100/300 h Lc,Ld cm", pp.L_charge_m*100, pp.L_discharge_m*100)
